# 🩺 Framingham Heart Study - Part I: Data Exploration & Feature Engineering

**10-Year Coronary Heart Disease Risk Prediction**

*Comprehensive exploratory data analysis with clinical domain expertise*

---

## Notebook Overview

This notebook performs detailed exploratory data analysis (EDA) on the Framingham Heart Study dataset, focusing on:

1. **Data Quality Assessment** - Missing values, outliers, data types
2. **Univariate Analysis** - Distribution of each feature
3. **Bivariate Analysis** - Feature relationships with target variable
4. **Clinical Feature Engineering** - Guideline-based categories and derived features
5. **Multivariate Analysis** - Correlations and interactions
6. **Data Preparation** - Final dataset ready for modeling

---

## Dataset: Framingham Heart Study

**Objective:** Predict 10-year risk of coronary heart disease (CHD)

**Features (15):**
- Demographics: age, sex (male), education
- Behavioral: currentSmoker, cigsPerDay
- Medical History: prevalentStroke, prevalentHyp, diabetes, BPMeds
- Clinical Measurements: sysBP, diaBP, BMI, heartRate
- Laboratory: totChol, glucose

**Target:** TenYearCHD (0 = No CHD, 1 = CHD within 10 years)

## Section 1: Import Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(" Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Section 2: Data Loading & Initial Exploration

In [ ]:
df_raw = pd.read_csv('../data/framingham_heart_study.csv')

print(f" Dataset loaded successfully")
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\nMemory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
print("\n First 10 rows of the dataset:")
df_raw.head(10)

In [ ]:
print("\n Dataset Information:")
df_raw.info()

In [ ]:
print("\n Statistical Summary (Numerical Features):")
df_raw.describe()

### 2.1 Target Variable Analysis

In [ ]:
print("\n TARGET VARIABLE: TenYearCHD")
print("="*60)

target_counts = df_raw['TenYearCHD'].value_counts()
target_pct = df_raw['TenYearCHD'].value_counts(normalize=True) * 100

print(f"\nClass Distribution:")
print(f"  No CHD (0): {target_counts[0]:,} ({target_pct[0]:.2f}%)")
print(f"  CHD (1):    {target_counts[1]:,} ({target_pct[1]:.2f}%)")

imbalance_ratio = target_counts[0] / target_counts[1]
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")
print(f"\n This is a SEVERELY IMBALANCED dataset!")
print(f"   We'll need to use SMOTE and appropriate metrics (ROC-AUC, Sensitivity, Specificity)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

target_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Target Variable Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('TenYearCHD')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['No CHD (0)', 'CHD (1)'], rotation=0)
axes[0].grid(axis='y', alpha=0.3)

for i, v in enumerate(target_counts):
    axes[0].text(i, v + 50, f"{v:,}\n({target_pct[i]:.1f}%)", 
                ha='center', fontweight='bold')

colors = ['#2ecc71', '#e74c3c']
axes[1].pie(target_counts, labels=['No CHD', 'CHD'], autopct='%1.1f%%',
           startangle=90, colors=colors, textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[1].set_title('Target Variable Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/01_target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Plot saved: outputs/01_target_distribution.png")

### 2.2 Missing Data Analysis

In [ ]:
print("\n MISSING DATA ANALYSIS")
print("="*60)

missing_data = pd.DataFrame({
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Percentage': (df_raw.isnull().sum() / len(df_raw)) * 100
}).sort_values('Missing_Count', ascending=False)

missing_data = missing_data[missing_data['Missing_Count'] > 0]

if len(missing_data) > 0:
    print(f"\nFeatures with missing values: {len(missing_data)}/{df_raw.shape[1]}")
    print(f"\nTotal missing values: {df_raw.isnull().sum().sum():,}")
    print(f"\nDetailed breakdown:\n")
    for idx, row in missing_data.iterrows():
        print(f"  {idx:20s}: {row['Missing_Count']:4.0f} ({row['Missing_Percentage']:5.2f}%)")
else:
    print("\n No missing values found!")

In [ ]:
if len(missing_data) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    missing_data_plot = missing_data.sort_values('Missing_Percentage', ascending=True)
    bars = ax.barh(missing_data_plot.index, missing_data_plot['Missing_Percentage'], 
                   color='#e74c3c', alpha=0.7)
    
    ax.set_xlabel('Missing Percentage (%)', fontsize=12, fontweight='bold')
    ax.set_title('Missing Data by Feature', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    for i, (idx, row) in enumerate(missing_data_plot.iterrows()):
        ax.text(row['Missing_Percentage'] + 0.2, i, 
               f"{row['Missing_Percentage']:.1f}% ({row['Missing_Count']:.0f})",
               va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../outputs/02_missing_data.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(" Plot saved: outputs/02_missing_data.png")

## Section 3: Univariate Analysis

Detailed analysis of each feature's distribution

### 3.1 Categorical Features

In [ ]:
categorical_features = ['male', 'education', 'currentSmoker', 'BPMeds', 
                       'prevalentStroke', 'prevalentHyp', 'diabetes']

numerical_features = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 
                     'BMI', 'heartRate', 'glucose']

print(f"\n Feature Classification:")
print(f"  Categorical: {len(categorical_features)} features")
print(f"  Numerical:   {len(numerical_features)} features")
print(f"  Target:      1 feature (TenYearCHD)")

In [ ]:
print("\n CATEGORICAL FEATURES DISTRIBUTION")
print("="*80)

for feature in categorical_features:
    print(f"\n{feature.upper()}:")
    value_counts = df_raw[feature].value_counts(dropna=False)
    value_pct = df_raw[feature].value_counts(dropna=False, normalize=True) * 100
    
    for val, count in value_counts.items():
        print(f"  {val}: {count:,} ({value_pct[val]:.1f}%)")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.ravel()

for idx, feature in enumerate(categorical_features):
    value_counts = df_raw[feature].value_counts()
    
    axes[idx].bar(range(len(value_counts)), value_counts.values, 
                 color=sns.color_palette('husl', len(value_counts)))
    axes[idx].set_title(f'{feature}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Category')
    axes[idx].set_ylabel('Count')
    axes[idx].set_xticks(range(len(value_counts)))
    axes[idx].set_xticklabels(value_counts.index, rotation=45)
    axes[idx].grid(axis='y', alpha=0.3)
    
    for i, v in enumerate(value_counts.values):
        axes[idx].text(i, v + max(value_counts.values)*0.01, str(v), 
                      ha='center', fontsize=9)

axes[7].axis('off')

plt.suptitle('Categorical Features Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/03_categorical_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Plot saved: outputs/03_categorical_distributions.png")

### 3.2 Numerical Features

In [ ]:
print("\n NUMERICAL FEATURES STATISTICS")
print("="*80)

df_raw[numerical_features].describe().T

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 16))
axes = axes.ravel()

for idx, feature in enumerate(numerical_features):
    axes[idx].hist(df_raw[feature].dropna(), bins=30, color='skyblue', 
                  edgecolor='black', alpha=0.7, density=True)
    
    df_raw[feature].dropna().plot(kind='kde', ax=axes[idx], color='red', 
                                  linewidth=2, secondary_y=False)
    
    axes[idx].set_title(f'{feature}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Density')
    axes[idx].grid(alpha=0.3)
    
    mean_val = df_raw[feature].mean()
    median_val = df_raw[feature].median()
    axes[idx].axvline(mean_val, color='green', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    axes[idx].axvline(median_val, color='orange', linestyle='--', linewidth=2, label=f'Median: {median_val:.1f}')
    axes[idx].legend()

plt.suptitle('Numerical Features Distribution (Histogram + KDE)', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../outputs/04_numerical_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Plot saved: outputs/04_numerical_distributions.png")

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 16))
axes = axes.ravel()

for idx, feature in enumerate(numerical_features):
    sns.boxplot(data=df_raw, y=feature, ax=axes[idx], color='lightcoral')
    axes[idx].set_title(f'{feature} - Outlier Detection', fontsize=12, fontweight='bold')
    axes[idx].grid(axis='y', alpha=0.3)
    
    Q1 = df_raw[feature].quantile(0.25)
    Q3 = df_raw[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_raw[(df_raw[feature] < lower_bound) | (df_raw[feature] > upper_bound)][feature]
    outlier_pct = (len(outliers) / len(df_raw)) * 100
    
    axes[idx].text(0.5, 0.95, f'Outliers: {len(outliers)} ({outlier_pct:.1f}%)',
                  transform=axes[idx].transAxes, ha='center', va='top',
                  bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Numerical Features - Outlier Detection (Boxplots)', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../outputs/05_numerical_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Plot saved: outputs/05_numerical_boxplots.png")

## Section 4: Bivariate Analysis - Features vs Target

Analyze the relationship between each feature and the target variable (TenYearCHD)

### 4.1 Categorical Features vs Target

In [ ]:
print("\n CATEGORICAL FEATURES vs TARGET (TenYearCHD)")
print("="*80)

for feature in categorical_features:
    print(f"\n{feature.upper()}:")
    
    crosstab = pd.crosstab(df_raw[feature], df_raw['TenYearCHD'], margins=True, normalize='index')
    print(crosstab * 100)
    
    contingency_table = pd.crosstab(df_raw[feature], df_raw['TenYearCHD'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    print(f"\n  Chi-square test:")
    print(f"    χ² = {chi2:.4f}")
    print(f"    p-value = {p_value:.4f}")
    
    if p_value < 0.05:
        print(f"     SIGNIFICANT association with target (p < 0.05)")
    else:
        print(f"     No significant association with target (p >= 0.05)")
    print("    " + "-"*60)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for idx, feature in enumerate(categorical_features):
    crosstab = pd.crosstab(df_raw[feature], df_raw['TenYearCHD'], normalize='index') * 100
    crosstab.plot(kind='bar', ax=axes[idx], color=['#2ecc71', '#e74c3c'], alpha=0.8)
    
    axes[idx].set_title(f'{feature} vs TenYearCHD', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('Percentage (%)')
    axes[idx].legend(['No CHD', 'CHD'], loc='upper right')
    axes[idx].grid(axis='y', alpha=0.3)
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45)

axes[7].axis('off')

plt.suptitle('Categorical Features vs Target Variable', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../outputs/06_categorical_vs_target.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Plot saved: outputs/06_categorical_vs_target.png")

### 4.2 Numerical Features vs Target

In [ ]:
print("\n NUMERICAL FEATURES vs TARGET (TenYearCHD)")
print("="*80)

for feature in numerical_features:
    print(f"\n{feature.upper()}:")
    
    no_chd = df_raw[df_raw['TenYearCHD'] == 0][feature].dropna()
    chd = df_raw[df_raw['TenYearCHD'] == 1][feature].dropna()
    
    print(f"  No CHD - Mean: {no_chd.mean():.2f}, Median: {no_chd.median():.2f}, Std: {no_chd.std():.2f}")
    print(f"  CHD    - Mean: {chd.mean():.2f}, Median: {chd.median():.2f}, Std: {chd.std():.2f}")
    
    statistic, p_value = mannwhitneyu(no_chd, chd, alternative='two-sided')
    
    print(f"\n  Mann-Whitney U test:")
    print(f"    U-statistic = {statistic:.2f}")
    print(f"    p-value = {p_value:.6f}")
    
    if p_value < 0.05:
        print(f"     SIGNIFICANT difference between groups (p < 0.05)")
    else:
        print(f"     No significant difference between groups (p >= 0.05)")
    print("    " + "-"*60)

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(15, 18))
axes = axes.ravel()

for idx, feature in enumerate(numerical_features):
    sns.violinplot(data=df_raw, x='TenYearCHD', y=feature, ax=axes[idx],
                  palette=['#2ecc71', '#e74c3c'])
    
    axes[idx].set_title(f'{feature} by TenYearCHD', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('TenYearCHD')
    axes[idx].set_xticklabels(['No CHD', 'CHD'])
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Numerical Features vs Target (Violin Plots)', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('../outputs/07_numerical_vs_target.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Plot saved: outputs/07_numerical_vs_target.png")

## Section 5: Clinical Feature Engineering

Create clinically meaningful features based on medical guidelines:
- **AHA/ACC Guidelines**: Blood pressure categories
- **WHO Guidelines**: BMI classification  
- **NCEP ATP III**: Cholesterol risk categories
- **ADA Guidelines**: Glucose/diabetes risk
- **Derived Features**: Pack-years, pulse pressure, interaction terms

In [ ]:
df_processed = df_raw.copy()

print(" Starting Clinical Feature Engineering")
print("="*80)

### 5.1 Age Stratification (Cardiovascular Risk Groups)

In [ ]:
def create_age_groups(age):
    """Categorize age into cardiovascular risk groups."""
    if age < 40:
        return 0
    elif age < 55:
        return 1
    elif age < 65:
        return 2
    else:
        return 3

df_processed['age_group'] = df_processed['age'].apply(create_age_groups)

print("\n Age Groups Created:")
print(df_processed['age_group'].value_counts().sort_index())
print("\n  0 = Young (<40)")
print("  1 = Middle-aged (40-54)")
print("  2 = Pre-elderly (55-64)")
print("  3 = Elderly (≥65)")

### 5.2 Blood Pressure Categories (AHA/ACC 2017 Guidelines)

In [ ]:
def bp_category(sysBP, diaBP):
    """Categorize blood pressure per AHA/ACC 2017 guidelines."""
    if sysBP < 120 and diaBP < 80:
        return 0
    elif sysBP < 130 and diaBP < 80:
        return 1
    elif sysBP < 140 or diaBP < 90:
        return 2
    else:
        return 3

df_processed['bp_category'] = df_processed.apply(
    lambda row: bp_category(row['sysBP'], row['diaBP']), axis=1
)

print("\n Blood Pressure Categories Created (AHA/ACC Guidelines):")
print(df_processed['bp_category'].value_counts().sort_index())
print("\n  0 = Normal (<120/<80 mmHg)")
print("  1 = Elevated (120-129/<80 mmHg)")
print("  2 = Stage 1 HTN (130-139/80-89 mmHg)")
print("  3 = Stage 2 HTN (≥140/≥90 mmHg)")

### 5.3 BMI Classification (WHO Guidelines)

In [ ]:
def bmi_category(bmi):
    """Categorize BMI per WHO guidelines."""
    if pd.isna(bmi):
        return np.nan
    elif bmi < 18.5:
        return 0
    elif bmi < 25:
        return 1
    elif bmi < 30:
        return 2
    else:
        return 3

df_processed['bmi_category'] = df_processed['BMI'].apply(bmi_category)

print("\n BMI Categories Created (WHO Guidelines):")
print(df_processed['bmi_category'].value_counts().sort_index())
print("\n  0 = Underweight (<18.5)")
print("  1 = Normal (18.5-24.9)")
print("  2 = Overweight (25-29.9)")
print("  3 = Obese (≥30)")

### 5.4 Cholesterol Risk Categories (NCEP ATP III)

In [ ]:
def chol_risk(totChol):
    """Categorize total cholesterol per NCEP ATP III guidelines."""
    if pd.isna(totChol):
        return np.nan
    elif totChol < 200:
        return 0
    elif totChol < 240:
        return 1
    else:
        return 2

df_processed['chol_risk'] = df_processed['totChol'].apply(chol_risk)

print("\n Cholesterol Risk Categories Created (NCEP ATP III):")
print(df_processed['chol_risk'].value_counts().sort_index())
print("\n  0 = Desirable (<200 mg/dL)")
print("  1 = Borderline High (200-239 mg/dL)")
print("  2 = High (≥240 mg/dL)")

### 5.5 Glucose/Diabetes Risk Categories

In [ ]:
def glucose_category(glucose, diabetes):
    """Categorize glucose levels and diabetes status."""
    if diabetes == 1:
        return 3
    elif pd.isna(glucose):
        return np.nan
    elif glucose < 100:
        return 0
    elif glucose < 126:
        return 1
    else:
        return 2

df_processed['glucose_category'] = df_processed.apply(
    lambda row: glucose_category(row['glucose'], row['diabetes']), axis=1
)

print("\n Glucose/Diabetes Categories Created:")
print(df_processed['glucose_category'].value_counts().sort_index())
print("\n  0 = Normal (<100 mg/dL)")
print("  1 = Pre-diabetic (100-125 mg/dL)")
print("  2 = Diabetic range (≥126 mg/dL, undiagnosed)")
print("  3 = Known diabetes")

### 5.6 Smoking Burden (Pack-Years)

In [ ]:
df_processed['pack_years'] = (df_processed['cigsPerDay'] / 20) * (df_processed['age'] - 18)
df_processed['pack_years'] = df_processed['pack_years'].clip(lower=0)

print("\n Pack-Years Calculated:")
print(f"  Mean: {df_processed['pack_years'].mean():.2f}")
print(f"  Median: {df_processed['pack_years'].median():.2f}")
print(f"  Max: {df_processed['pack_years'].max():.2f}")

### 5.7 Pulse Pressure (Arterial Stiffness Indicator)

In [ ]:
df_processed['pulse_pressure'] = df_processed['sysBP'] - df_processed['diaBP']

print("\n Pulse Pressure Calculated:")
print(f"  Mean: {df_processed['pulse_pressure'].mean():.2f} mmHg")
print(f"  Median: {df_processed['pulse_pressure'].median():.2f} mmHg")
print("\n  (Elevated pulse pressure >60 mmHg indicates arterial stiffness)")

### 5.8 Interaction Terms (Synergistic Risk Factors)

In [ ]:
df_processed['age_bmi_interaction'] = df_processed['age'] * df_processed['BMI'] / 100

df_processed['age_smoking_interaction'] = df_processed['age'] * df_processed['currentSmoker']

df_processed['bp_age_interaction'] = df_processed['sysBP'] * df_processed['age'] / 1000

df_processed['chol_age_interaction'] = df_processed['totChol'] * df_processed['age'] / 1000

print("\n Interaction Terms Created:")
print("  - age_bmi_interaction: Combined metabolic burden")
print("  - age_smoking_interaction: Cumulative tobacco exposure")
print("  - bp_age_interaction: Vascular aging effect")
print("  - chol_age_interaction: Lipid-age synergy")

In [ ]:
engineered_features = [
    'age_group', 'bp_category', 'bmi_category', 'chol_risk', 
    'glucose_category', 'pack_years', 'pulse_pressure',
    'age_bmi_interaction', 'age_smoking_interaction', 
    'bp_age_interaction', 'chol_age_interaction'
]

print("\n" + "="*80)
print(" FEATURE ENGINEERING SUMMARY")
print("="*80)
print(f"\nOriginal features: {df_raw.shape[1]}")
print(f"Engineered features: {len(engineered_features)}")
print(f"Total features: {df_processed.shape[1]}")
print(f"\nNew features added:")
for i, feat in enumerate(engineered_features, 1):
    print(f"  {i:2d}. {feat}")

## Section 6: Multivariate Analysis

### 6.1 Correlation Analysis

In [ ]:
numerical_cols_all = df_processed.select_dtypes(include=[np.number]).columns.tolist()

corr_matrix = df_processed[numerical_cols_all].corr()

plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
           center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})

plt.title('Correlation Matrix - All Numerical Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/08_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Plot saved: outputs/08_correlation_matrix.png")

In [ ]:
target_corr = corr_matrix['TenYearCHD'].sort_values(ascending=False)

print("\n TOP CORRELATIONS WITH TARGET (TenYearCHD):")
print("="*60)
print("\nPositive correlations (risk factors):")
print(target_corr[target_corr > 0].head(15))
print("\nNegative correlations (protective factors):")
print(target_corr[target_corr < 0].tail(10))

In [ ]:
plt.figure(figsize=(12, 10))
target_corr_abs = target_corr.abs().sort_values(ascending=True).tail(20)

colors = ['red' if target_corr[feat] > 0 else 'blue' for feat in target_corr_abs.index]
target_corr_abs.plot(kind='barh', color=colors, alpha=0.7)

plt.title('Top 20 Features by Correlation with Target', fontsize=14, fontweight='bold')
plt.xlabel('Absolute Correlation')
plt.ylabel('Feature')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/09_target_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Plot saved: outputs/09_target_correlations.png")

## Section 7: Data Quality & Missing Value Treatment

In [ ]:
print("\n MISSING VALUES IN PROCESSED DATA")
print("="*60)

missing_summary = pd.DataFrame({
    'Missing_Count': df_processed.isnull().sum(),
    'Missing_Percentage': (df_processed.isnull().sum() / len(df_processed)) * 100
}).sort_values('Missing_Count', ascending=False)

missing_summary = missing_summary[missing_summary['Missing_Count'] > 0]

if len(missing_summary) > 0:
    print(f"\nFeatures with missing values: {len(missing_summary)}")
    print(f"\nBreakdown:\n")
    print(missing_summary)
    print("\n Note: Missing values will be handled using KNN imputation during modeling")
else:
    print("\n No missing values!")

### 7.1 Clinical Plausibility Checks

In [ ]:
print("\n🩺 CLINICAL PLAUSIBILITY CHECKS")
print("="*60)

issues_found = []

invalid_bp = df_processed[df_processed['sysBP'] <= df_processed['diaBP']]
if len(invalid_bp) > 0:
    issues_found.append(f" {len(invalid_bp)} rows where sysBP ≤ diaBP (clinically impossible)")

if df_processed['age'].min() < 18 or df_processed['age'].max() > 120:
    issues_found.append(f" Age outside reasonable range: {df_processed['age'].min()}-{df_processed['age'].max()}")

if df_processed['BMI'].min() < 10 or df_processed['BMI'].max() > 70:
    issues_found.append(f" BMI outside normal range: {df_processed['BMI'].min():.1f}-{df_processed['BMI'].max():.1f}")

if df_processed['heartRate'].min() < 30 or df_processed['heartRate'].max() > 220:
    issues_found.append(f" Heart rate outside normal range: {df_processed['heartRate'].min():.0f}-{df_processed['heartRate'].max():.0f}")

if len(issues_found) > 0:
    print("\nIssues detected:")
    for issue in issues_found:
        print(f"  {issue}")
else:
    print("\n All values are clinically plausible!")

## Section 8: Final Dataset Preparation & Export

In [ ]:
print("\n" + "="*80)
print(" FINAL PROCESSED DATASET SUMMARY")
print("="*80)

print(f"\nDataset shape: {df_processed.shape[0]:,} rows × {df_processed.shape[1]} columns")
print(f"\nFeature breakdown:")
print(f"  - Original features: {len(df_raw.columns) - 1} (excluding target)")
print(f"  - Engineered features: {len(engineered_features)}")
print(f"  - Target variable: 1 (TenYearCHD)")
print(f"  - Total features: {df_processed.shape[1] - 1}")

print(f"\nTarget distribution:")
print(f"  - No CHD: {(df_processed['TenYearCHD'] == 0).sum():,} ({(df_processed['TenYearCHD'] == 0).sum() / len(df_processed) * 100:.1f}%)")
print(f"  - CHD: {(df_processed['TenYearCHD'] == 1).sum():,} ({(df_processed['TenYearCHD'] == 1).sum() / len(df_processed) * 100:.1f}%)")

In [ ]:
print("\n ALL FEATURES IN PROCESSED DATASET:")
print("="*60)

all_features = [col for col in df_processed.columns if col != 'TenYearCHD']

print(f"\nOriginal Features ({len(df_raw.columns) - 1}):")
for i, feat in enumerate([c for c in df_raw.columns if c != 'TenYearCHD'], 1):
    print(f"  {i:2d}. {feat}")

print(f"\nEngineered Features ({len(engineered_features)}):")
for i, feat in enumerate(engineered_features, 1):
    print(f"  {i:2d}. {feat}")

print(f"\nTarget Variable:")
print(f"  1. TenYearCHD")

In [ ]:
output_path = '../data/framingham_processed.csv'
df_processed.to_csv(output_path, index=False)

print(f"\n Processed dataset saved to: {output_path}")
print(f"\nFile size: {df_processed.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## Summary & Key Insights

### Dataset Characteristics:
- **Highly imbalanced**: ~85% no CHD, ~15% CHD (will use SMOTE for training)
- **Missing data**: Present in glucose, education, cholesterol, BPMeds (will use KNN imputation)
- **Feature-rich**: Combined original + engineered features for robust modeling

### Clinical Feature Engineering Applied:
1. **Age Groups**: Cardiovascular risk stratification
2. **BP Categories**: AHA/ACC 2017 guidelines
3. **BMI Categories**: WHO classification
4. **Cholesterol Risk**: NCEP ATP III guidelines
5. **Glucose Categories**: Diabetes risk assessment
6. **Pack-Years**: Smoking exposure quantification
7. **Pulse Pressure**: Arterial stiffness marker
8. **Interaction Terms**: Synergistic risk factors

### Next Steps (Notebook 02):
1. ✅ **Data prepared and saved**
2. → **Train-test split** (stratified, 80/20)
3. → **SMOTE** for class imbalance
4. → **Hyperparameter optimization** (Optuna, 50+ trials)
5. → **Model training** (XGBoost, RandomForest, Ensemble)
6. → **Clinical evaluation** (ROC-AUC, Sensitivity, Specificity, PPV, NPV)
7. → **SHAP analysis** (global + patient-level explanations)

---

**Notebook 01 Complete! Ready for Model Training.**